<a href="https://colab.research.google.com/github/SharfSid18/AI-Chatbot/blob/main/FINAL_AI_CHATBOT_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio transformers duckduckgo-search --upgrade

from transformers import AutoModelForCausalLM, AutoTokenizer
from duckduckgo_search import DDGS
import gradio as gr
import torch

# Load HuggingFace DialoGPT model
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

chat_history_ids = None  # For conversation continuity

# Check if input is a factual question
def is_factual_question(text):
    keywords = ["what is", "who is", "define", "explain", "difference between", "how does", "tell me about"]
    return any(text.lower().startswith(k) for k in keywords)

# Get real-time answer from DuckDuckGo
def search_real_time(query):
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=1)
        if results:
            return results[0]["body"]
        else:
            return "Sorry, I couldn't find a real-time answer. Try rephrasing your question."

# Main chatbot response logic
def chatbot_response(user_input, history=[]):
    global chat_history_ids

    if is_factual_question(user_input):
        response = search_real_time(user_input)
    else:
        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8
        )
        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    history.append((f"🧑 You: {user_input}", f"🤖 StarBot: {response}"))
    return history, history, ""

# Reset Chat
def clear_chat():
    global chat_history_ids
    chat_history_ids = None
    return [], [], ""

# Download Chat
def download_chat(history):
    text = "\n\n".join([f"{x[0]}\n{x[1]}" for x in history])
    path = "chat_history.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path

# CSS - Dark Mode
custom_css = """
body {
    background-color: #000000;
    color: white;
    font-family: 'Segoe UI', sans-serif;
}
.gradio-container {
    background-color: #000000;
}
h2, .gr-markdown {
    color: white;
    text-align: center;
    font-size: 28px;
    margin-bottom: 20px;
}
.message {
    background-color: #111111 !important;
    color: white !important;
    border-radius: 10px;
    padding: 10px 16px;
    margin: 5px 0;
    font-size: 16px;
}
#chatbot .message.user {
    background-color: #222222 !important;
    color: #ffffff !important;
    border-left: 4px solid #00bfff;
}
#chatbot .message.bot {
    background-color: #333333 !important;
    color: #ffffff !important;
    border-left: 4px solid #00ff99;
}

/* Buttons Styling */
button {
    font-weight: bold;
    border-radius: 12px;
    padding: 10px 20px;
    font-size: 16px;
    border: none;
    transition: all 0.3s ease-in-out;
}

/* Send Button (Green) */
button:nth-of-type(1) {
    background-color: #28a745 !important;
    color: white !important;
}
button:nth-of-type(1):hover {
    background-color: #218838 !important;
}

/* Clear Chat Button (Red) */
button:nth-of-type(2) {
    background-color: #dc3545 !important;
    color: white !important;
}
button:nth-of-type(2):hover {
    background-color: #c82333 !important;
}

/* Download Chat Button (Blue) */
button:nth-of-type(3) {
    background-color: #007bff !important;
    color: white !important;
}
button:nth-of-type(3):hover {
    background-color: #0056b3 !important;
}

/* Input Fields */
input, textarea {
    background-color: #111111 !important;
    color: white !important;
    border: 1px solid #ffffff44 !important;
    border-radius: 12px;
    padding: 10px;
    font-size: 16px;
}
"""


# Gradio UI
with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("<h2>🤖 StarBot </h2>")
    chatbot = gr.Chatbot(elem_id="chatbot", label="Talk to StarBot")

    with gr.Row():
        msg = gr.Textbox(placeholder="Say 'Hi' or ask 'How are you?'", show_label=False, scale=6)
        send_btn = gr.Button("Send", scale=1)

    with gr.Row():
        clear_btn = gr.Button("🧹 Clear Chat")
        download_btn = gr.Button("⬇️ Download Chat")

    state = gr.State([])

    msg.submit(chatbot_response, [msg, state], [chatbot, state, msg])
    send_btn.click(chatbot_response, [msg, state], [chatbot, state, msg])
    clear_btn.click(clear_chat, outputs=[chatbot, state, msg])
    download_btn.click(download_chat, inputs=state, outputs=gr.File())

demo.launch()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.49.0
    Uninstalling transformers-4.49.0:
      Successfully uninstalled transformers-4.49.0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

<ipython-input-1-c733ea564a7d>:156: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(elem_id="chatbot", label="Talk to StarBot")


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c25873a2de871b5d8.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
